In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from joblib import dump

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split


from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, Dropout, GRU , Bidirectional,BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

c:\Users\Amir sohail\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
df = pd.read_csv("flipkart_product.csv",encoding="latin1")
df.head()

,ProductName,Price,Rate,Review,Summary
0,Candes 12 L Room/Personal Air Cooler?ÿ?ÿ(White...,"??3,999",5,Super!,Great cooler.. excellent air flow and for this...
1,Candes 12 L Room/Personal Air Cooler?ÿ?ÿ(White...,"??3,999",5,Awesome,Best budget 2 fit cooler. Nice cooling
2,Candes 12 L Room/Personal Air Cooler?ÿ?ÿ(White...,"??3,999",3,Fair,The quality is good but the power of air is de...
3,Candes 12 L Room/Personal Air Cooler?ÿ?ÿ(White...,"??3,999",1,Useless product,Very bad product it's a only a fan
4,Candes 12 L Room/Personal Air Cooler?ÿ?ÿ(White...,"??3,999",3,Fair,Ok ok product


In [3]:
df.drop(["ProductName","Price"],axis=1,inplace=True)
df.head()

,Rate,Review,Summary
0,5,Super!,Great cooler.. excellent air flow and for this...
1,5,Awesome,Best budget 2 fit cooler. Nice cooling
2,3,Fair,The quality is good but the power of air is de...
3,1,Useless product,Very bad product it's a only a fan
4,3,Fair,Ok ok product


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 189874 entries, 0 to 189873
Data columns (total 3 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   Rate     189873 non-null  object
 1   Review   189870 non-null  object
 2   Summary  189860 non-null  object
dtypes: object(3)
memory usage: 4.3+ MB


In [5]:
df.drop_duplicates(inplace=True)

In [6]:
df.duplicated().sum()

np.int64(0)

In [7]:
df.shape

(116941, 3)

In [8]:
df.columns

Index(['Rate', 'Review', 'Summary'], dtype='object')

In [9]:
df.head()

,Rate,Review,Summary
0,5,Super!,Great cooler.. excellent air flow and for this...
1,5,Awesome,Best budget 2 fit cooler. Nice cooling
2,3,Fair,The quality is good but the power of air is de...
3,1,Useless product,Very bad product it's a only a fan
4,3,Fair,Ok ok product


In [10]:
df["Rate"].value_counts()

Rate
5                                                              64289
4                                                              22198
1                                                              15419
3                                                               9988
2                                                               5042
Pigeon Favourite Electric Kettle?ÿ?ÿ(1.5 L, Silver, Black)         1
Bajaj DX 2 L/W Dry Iron                                            1
Nova Plus Amaze NI 10 1100 W Dry Iron?ÿ?ÿ(Grey & Turquoise)        1
s                                                                  1
Name: count, dtype: int64

In [11]:
df.dropna(inplace=True)
df.isnull().sum()

Rate       0
Review     0
Summary    0
dtype: int64

In [12]:
df["Rate"].unique()

array(['5', '3', '1', '4', '2',
       'Pigeon Favourite Electric Kettle?ÿ?ÿ(1.5 L, Silver, Black)',
       'Bajaj DX 2 L/W Dry Iron',
       'Nova Plus Amaze NI 10 1100 W Dry Iron?ÿ?ÿ(Grey & Turquoise)', 's'],
      dtype=object)

In [13]:
def Rate_(rate):
    l = ["Pigeon Favourite Electric Kettle?ÿ?ÿ(1.5 L, Silver, Black)","Bajaj DX 2 L/W Dry Iron","Nova Plus Amaze NI 10 1100 W Dry Iron?ÿ?ÿ(Grey & Turquoise)","s"]
    if rate in l:
        return 2
    return int(rate)

# Rate_("1")
df["Rate"] = df["Rate"].apply(Rate_)
df.head()

,Rate,Review,Summary
0,5,Super!,Great cooler.. excellent air flow and for this...
1,5,Awesome,Best budget 2 fit cooler. Nice cooling
2,3,Fair,The quality is good but the power of air is de...
3,1,Useless product,Very bad product it's a only a fan
4,3,Fair,Ok ok product


In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 116932 entries, 0 to 189867
Data columns (total 3 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   Rate     116932 non-null  int64 
 1   Review   116932 non-null  object
 2   Summary  116932 non-null  object
dtypes: int64(1), object(2)
memory usage: 3.6+ MB


In [15]:
df["Rate"].value_counts()

Rate
5    64286
4    22195
1    15419
3     9987
2     5045
Name: count, dtype: int64

In [16]:
df["Text"] = df["Review"] + " " + df["Summary"] 
df.drop(["Review",'Summary'],axis=1,inplace=True)
df.head()

,Rate,Text
0,5,Super! Great cooler.. excellent air flow and f...
1,5,Awesome Best budget 2 fit cooler. Nice cooling
2,3,Fair The quality is good but the power of air ...
3,1,Useless product Very bad product it's a only a...
4,3,Fair Ok ok product


In [17]:
import string

def remove_punc(txt):
  return txt.translate(str.maketrans('','',string.punctuation))

def tolower(txt):
    return txt.lower()

def remove_num(txt):
    new = ""
    for i in txt:
        if not i.isdigit():
            new+=i
    return new

def remove_emoj(txt):
    new = ""
    for i in txt:
        if i.isascii():
            new+=i
    return new

In [18]:
df["Text"] = df["Text"].apply(remove_punc)
df["Text"] = df["Text"].apply(tolower)
df["Text"] = df["Text"].apply(remove_num)
df["Text"] = df["Text"].apply(remove_emoj)

df.head()

,Rate,Text
0,5,super great cooler excellent air flow and for ...
1,5,awesome best budget fit cooler nice cooling
2,3,fair the quality is good but the power of air ...
3,1,useless product very bad product its a only a fan
4,3,fair ok ok product


In [19]:
Max_len = 100
Max_words = 10000
Embedding_dim = 100

In [20]:
X = df.drop("Rate",axis=1)
y = df["Rate"]

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [21]:
tokenizer = Tokenizer(num_words=5000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train["Text"])
train_seq = tokenizer.texts_to_sequences(X_train["Text"])
train_X_pad = pad_sequences(train_seq, maxlen=Max_len, padding="post", truncating="post")
test_seq = tokenizer.texts_to_sequences(X_test["Text"])
test_X_pad = pad_sequences(test_seq, maxlen=Max_len, padding="post",truncating="post")

In [22]:
len(df["Rate"].unique())

5

In [23]:
model = Sequential([
    Embedding(input_dim=Max_words, output_dim=Embedding_dim, input_length=Max_len),
    Dense(128,activation="relu"),
    BatchNormalization(),
    Bidirectional(GRU(64,return_sequences=True)),
    Dense(128,activation="relu"),
    BatchNormalization(),
    Bidirectional(GRU(64,return_sequences=True)),
    Dense(32,activation="relu"),
    BatchNormalization(),
    Bidirectional(GRU(64,return_sequences=False)),
    Dropout(0.3),
    Dense(32,activation="relu"),
    BatchNormalization(),
    Dropout(0.2),
    Dense(16,activation="relu"),
    BatchNormalization(),
    Dropout(0.1),
    Dense(8,activation="relu"),
    BatchNormalization(),
    Dense(5,activation="softmax")
])

c:\Users\Amir sohail\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [24]:
# model.compile(loss="mae", optimizer="adam", metrics=["r2_score"])
# model.EarlyStopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
# his = model.fit(train_X_pad, y_train, validation_data=(test_X_pad, y_test), epochs=2, batch_size=256)
# model.summary()

In [25]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier,GradientBoostingClassifier,ExtraTreesClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

In [26]:
models = {
    # "logistic_regression": LogisticRegression(),
    # "Decision_tree": DecisionTreeClassifier(),
    # "knn": KNeighborsClassifier(),
    # "AdaBoostClassifier": AdaBoostClassifier(),
    # "XGBClassifier": XGBClassifier(),
    'Extra Trees': ExtraTreesClassifier(random_state=42, n_jobs=-1),
    # "GradientBoostingClassifier": GradientBoostingClassifier(),
    # "RandomForestClassifier": RandomForestClassifier()
}

In [27]:
results = []

for name,model in models.items():
    model.fit(train_X_pad,y_train)
    y_pred = model.predict(test_X_pad)
    y_pred_train = model.predict(train_X_pad)
    acc_test = accuracy_score(y_test,y_pred)
    acc_train = accuracy_score(y_train,y_pred_train)
    results.append({
        'model':name,
        'accuracy_test':acc_test,
        'accuracy_train':acc_train
    })
df_results = pd.DataFrame(results)

df_results

,model,accuracy_test,accuracy_train
0,Extra Trees,0.832343,0.987022


In [28]:
final_model = models["Extra Trees"]
y_pred = model.predict(test_X_pad)
accuracy_score(y_test,y_pred)

0.8323427545217429

In [29]:
from joblib import dump

dump(final_model,"model.pkl")
dump(tokenizer,"tokenizer.pkl")
dump(X.columns.tolist(),"columns.pkl")

['columns.pkl']